<a href="https://colab.research.google.com/github/danisxde/x5_ml/blob/main/x5_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

# Загрузка данных
df = pd.read_csv('/content/drive/MyDrive/train_2.csv')

rto_09 = df.loc[(df['new_id'] == 9754) & (df['Год'] == 2023) & (df['Месяц'] == 9), 'РТО'].values[0]
rto_11 = df.loc[(df['new_id'] == 9754) & (df['Год'] == 2023) & (df['Месяц'] == 11), 'РТО'].values[0]

mean_rto = (rto_09 + rto_11) / 2

df.loc[(df['new_id'] == 9754) & (df['Год'] == 2023) & (df['Месяц'] == 10), 'РТО'] = mean_rto

# Создаём дату
df['date'] = pd.to_datetime(df['Год'].astype(str) + '-' + df['Месяц'].astype(str) + '-01')

# Переименовываем колонки для удобства
df = df.rename(columns={
    'Среднее количество промо товаров в чеке': 'avg_promo',
    'Среднее количество товаров в чеке': 'avg_items',
    'Среднее количество отмен': 'avg_cancels',
    'Рабочие часы в день': 'work_hours',
    'Дата открытия, категориальный': 'store_age_cat',
    'Торговая площадь, категориальный': 'area_cat',
    'Населенный пункт': 'city',
    'Регион': 'region',
    'Численность населения': 'population',
    'Количество домохозяйств': 'households',
    'Трафик пеший, в час': 'foot_traffic',
    'Трафик авто, в час': 'car_traffic',
    'Маркетплейсы, доставки, постаматы (100 м)': 'marketplaces',
    'Медицинские уч. и аптеки (300 м)': 'medical',
    'Школы (300 м)': 'schools',
    'Остановки (300 м)': 'stops',
    'Продуктовые магазины (500 м)': 'grocery_competitors',
    'Пятерочки (500 м)': 'pyaterochka_nearby',
    'Количество касс': 'n_kass',
    'Флаг алкогольной лицензии': 'alcohol_flag',
    'РТО': 'rto'
})

# Сортировка — критически важно для лагов
df = df.sort_values(['new_id', 'date']).reset_index(drop=True)

# Проверка
print(f"Shape: {df.shape}")
print(f"Даты: {df['date'].min()} — {df['date'].max()}")
print(f"Магазинов: {df['new_id'].nunique()}")
print(f"\nПервые строки:")
df.head()

Mounted at /content/drive
Shape: (485082, 25)
Даты: 2023-01-01 00:00:00 — 2025-02-01 00:00:00
Магазинов: 18657

Первые строки:


,new_id,Год,Месяц,avg_promo,avg_items,avg_cancels,work_hours,store_age_cat,area_cat,city,...,marketplaces,medical,schools,stops,grocery_competitors,pyaterochka_nearby,n_kass,alcohol_flag,rto,date
0,0,2023,1,1.32,6.04,162.0,16.0,Новый,Большой,Ярославль г,...,1,0,0,0,3,1,10,1,74914754.22,2023-01-01
1,0,2023,2,1.37,5.89,118.0,16.0,Новый,Большой,Ярославль г,...,1,0,0,0,3,1,10,1,69240001.40,2023-02-01
2,0,2023,3,1.34,5.87,236.0,16.0,Новый,Большой,Ярославль г,...,1,0,0,0,3,1,10,1,79905726.38,2023-03-01
3,0,2023,4,1.55,5.89,174.0,16.0,Новый,Большой,Ярославль г,...,1,0,0,0,3,1,10,1,78567643.63,2023-04-01
4,0,2023,5,1.79,5.70,218.0,16.0,Новый,Большой,Ярославль г,...,1,0,0,0,3,1,10,1,80856174.96,2023-05-01


In [2]:
eps = 1e-6

# =========================
# 1) Календарные признаки
# =========================
df['year'] = df['Год'].astype(np.int16)
df['month'] = df['Месяц'].astype(np.int8)

df['time_idx'] = (df['year'] - df['year'].min()) * 12 + (df['month'] - 1)
df['quarter'] = ((df['month'] - 1) // 3 + 1).astype(np.int8)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df['is_december'] = (df['month'] == 12).astype(np.int8)
df['is_q4'] = df['quarter'].isin([4]).astype(np.int8)

# =========================
# 2) Ordinal encoding для естественно упорядоченных категорий
# =========================
store_age_map = {
    'Новый': 0,
    'Средний по возрасту': 1,
    'Открыт давно': 2
}

area_map = {
    'Маленький': 0,
    'Средний': 1,
    'Большой': 2,
    'Очень большой': 3
}

df['store_age_ord'] = df['store_age_cat'].map(store_age_map).astype(np.int8)
df['area_ord'] = df['area_cat'].map(area_map).astype(np.int8)

# Для CatBoost оставим исходные как category
for col in ['city', 'region', 'store_age_cat', 'area_cat']:
    df[col] = df[col].astype('category')

# =========================
# 3) Frequency encoding для высококардинальных категорий
# =========================
for col in ['city', 'region']:
    freq = df[col].value_counts(dropna=False)
    df[f'{col}_freq'] = df[col].map(freq).astype(np.int32)

# =========================
# 4) Лог-преобразования для сильно скошенных числовых статических признаков
# =========================
log_cols = [
    'population',
    'households',
    'foot_traffic',
    'car_traffic',
    'marketplaces',
    'medical',
    'schools',
    'stops',
    'grocery_competitors',
    'pyaterochka_nearby',
    'n_kass'
]

for col in log_cols:
    df[f'log1p_{col}'] = np.log1p(df[col].clip(lower=0))

# =========================
# 5) Динамические признаки: лаги по магазину
#    Используем только прошлые месяцы
# =========================
dyn_cols = ['rto', 'avg_promo', 'avg_items', 'avg_cancels', 'work_hours']
lag_list = [1, 2, 3, 6, 12, 24]

for col in dyn_cols:
    grp = df.groupby('new_id', sort=False)[col]

    # Лаги
    for lag in lag_list:
        df[f'{col}_lag_{lag}'] = grp.shift(lag)

    # Rolling statistics по прошлым значениям
    shifted = grp.shift(1)
    for win in [3, 6, 12]:
        df[f'{col}_roll_mean_{win}'] = shifted.groupby(df['new_id'], sort=False).transform(
            lambda x: x.rolling(win, min_periods=1).mean()
        )
        df[f'{col}_roll_std_{win}'] = shifted.groupby(df['new_id'], sort=False).transform(
            lambda x: x.rolling(win, min_periods=1).std(ddof=0)
        )
        df[f'{col}_roll_median_{win}'] = shifted.groupby(df['new_id'], sort=False).transform(
            lambda x: x.rolling(win, min_periods=1).median()
        )
        df[f'{col}_roll_min_{win}'] = shifted.groupby(df['new_id'], sort=False).transform(
            lambda x: x.rolling(win, min_periods=1).min()
        )
        df[f'{col}_roll_max_{win}'] = shifted.groupby(df['new_id'], sort=False).transform(
            lambda x: x.rolling(win, min_periods=1).max()
        )

# =========================
# 6) Сильные derived-features для RTO
# =========================
df['rto_diff_1'] = df['rto_lag_1'] - df['rto_lag_2']
df['rto_diff_3'] = df['rto_lag_1'] - df['rto_lag_3']
df['rto_diff_12'] = df['rto_lag_1'] - df['rto_lag_12']
df['rto_diff_24'] = df['rto_lag_12'] - df['rto_lag_24']

df['rto_pct_change_1'] = df['rto_diff_1'] / (df['rto_lag_2'].abs() + eps)
df['rto_pct_change_12'] = df['rto_diff_12'] / (df['rto_lag_12'].abs() + eps)
df['rto_pct_change_24'] = df['rto_diff_24'] / (df['rto_lag_24'].abs() + eps)

df['rto_ratio_to_3m_mean'] = df['rto_lag_1'] / (df['rto_roll_mean_3'] + eps)
df['rto_ratio_to_6m_mean'] = df['rto_lag_1'] / (df['rto_roll_mean_6'] + eps)
df['rto_ratio_to_12m_mean'] = df['rto_lag_1'] / (df['rto_roll_mean_12'] + eps)

df['rto_ratio_mom'] = df['rto_lag_1'] / (df['rto_lag_2'] + eps)
df['rto_ratio_yoy'] = df['rto_lag_1'] / (df['rto_lag_12'] + eps)
df['rto_ratio_2y'] = df['rto_lag_12'] / (df['rto_lag_24'] + eps)

# Экспоненциальное сглаживание
df['rto_ewm_3'] = df.groupby('new_id', sort=False)['rto'].transform(
    lambda x: x.shift(1).ewm(span=3, adjust=False).mean()
)
df['rto_ewm_6'] = df.groupby('new_id', sort=False)['rto'].transform(
    lambda x: x.shift(1).ewm(span=6, adjust=False).mean()
)

# Расширяющееся среднее / std
df['rto_exp_mean'] = df.groupby('new_id', sort=False)['rto'].transform(
    lambda x: x.shift(1).expanding(min_periods=1).mean()
)
df['rto_exp_std'] = df.groupby('new_id', sort=False)['rto'].transform(
    lambda x: x.shift(1).expanding(min_periods=2).std()
)

# =========================
# 7) Несколько полезных derived-фич для динамических операционных метрик
# =========================
for col in ['avg_promo', 'avg_items', 'avg_cancels', 'work_hours']:
    df[f'{col}_mom_ratio'] = df[f'{col}_lag_1'] / (df[f'{col}_lag_2'] + eps)
    df[f'{col}_yoy_ratio'] = df[f'{col}_lag_1'] / (df[f'{col}_lag_12'] + eps)
    df[f'{col}_to_3m_mean'] = df[f'{col}_lag_1'] / (df[f'{col}_roll_mean_3'] + eps)

# =========================
# 8) Короткая проверка
# =========================
new_feat_cols = [
    c for c in df.columns
    if (
        'lag_' in c or 'roll_' in c or 'ewm_' in c or 'exp_' in c or
        c.endswith('_ord') or c.endswith('_freq') or
        c.startswith('log1p_') or c in ['time_idx', 'quarter', 'month_sin', 'month_cos', 'is_december', 'is_q4']
    )
]

print("Новых признаков:", len(new_feat_cols))
print("Пример новых признаков:")
print(df[new_feat_cols].head(3).T.head(40))

Новых признаков: 130
Пример новых признаков:
                                     0             1             2
time_idx                      0.000000  1.000000e+00  2.000000e+00
quarter                       1.000000  1.000000e+00  1.000000e+00
month_sin                     0.500000  8.660254e-01  1.000000e+00
month_cos                     0.866025  5.000000e-01  6.123234e-17
is_december                   0.000000  0.000000e+00  0.000000e+00
is_q4                         0.000000  0.000000e+00  0.000000e+00
store_age_ord                 0.000000  0.000000e+00  0.000000e+00
area_ord                      2.000000  2.000000e+00  2.000000e+00
city_freq                  3146.000000  3.146000e+03  3.146000e+03
region_freq                6578.000000  6.578000e+03  6.578000e+03
log1p_population             13.311137  1.331114e+01  1.331114e+01
log1p_households              8.236421  8.236421e+00  8.236421e+00
log1p_foot_traffic            4.934474  4.934474e+00  4.934474e+00
log1p_car_traffic

In [3]:
# =========================
# 1) Производные статические признаки
# =========================

# Отношения РТО к характеристикам локации (используем lag_1, чтобы не было leakage)
df['rto_per_capita'] = df['rto_lag_1'] / (df['population'] + 1)
df['rto_per_household'] = df['rto_lag_1'] / (df['households'] + 1)
df['rto_per_kassa'] = df['rto_lag_1'] / (df['n_kass'] + 1)

# Конкурентная среда
df['pyaterochka_share'] = df['pyaterochka_nearby'] / (df['grocery_competitors'] + 1)
df['competitors_per_capita'] = df['grocery_competitors'] / (df['population'] + 1)

# Трафик
df['traffic_total'] = df['foot_traffic'] + df['car_traffic']
df['traffic_per_kassa'] = df['traffic_total'] / (df['n_kass'] + 1)
df['foot_traffic_share'] = df['foot_traffic'] / (df['traffic_total'] + 1)

# Инфраструктура (интегральный показатель «проходимости» локации)
df['infra_score'] = (
    df['marketplaces'] + df['medical'] + df['schools'] + df['stops']
)
df['log1p_infra_score'] = np.log1p(df['infra_score'])
df['infra_per_capita'] = df['infra_score'] / (df['population'] + 1)

# Площадь × трафик (прокси «потенциала» точки)
df['area_x_traffic'] = df['area_ord'] * df['traffic_total']
df['area_x_kass'] = df['area_ord'] * df['n_kass']
df['kass_per_area'] = df['n_kass'] / (df['area_ord'] + 1)

# =========================
# 2) Агрегаты по региону (на основе lag_1, без leakage)
# =========================
for agg_col in ['region', 'city']:
    grp = df.groupby([agg_col, 'date'])['rto_lag_1']

    df[f'{agg_col}_rto_median'] = grp.transform('median')
    df[f'{agg_col}_rto_mean'] = grp.transform('mean')
    df[f'{agg_col}_rto_std'] = grp.transform('std')

    # Отклонение магазина от медианы по группе
    df[f'{agg_col}_rto_dev'] = df['rto_lag_1'] - df[f'{agg_col}_rto_median']

    # Ранг магазина внутри группы (перцентиль)
    df[f'{agg_col}_rto_rank'] = grp.rank(pct=True)

# =========================
# 3) Агрегаты по (регион × месяц) — сезонность региона
# =========================
region_month = df.groupby(['region', 'month'])['rto_lag_1']
df['region_month_median'] = region_month.transform('median')
df['region_month_dev'] = df['rto_lag_1'] - df['region_month_median']

# =========================
# 4) Target Encoding: регион и город
#    Считаем на ПОЛНЫХ данных (потом при обучении будем делать OOF)
#    Сейчас — грубый вариант для feature engineering
#    Точный OOF target encoding сделаем в блоке обучения
# =========================

# Пока используем простой вариант — среднее РТО по группе
# (для бустингов это уже полезно, а OOF-версию добавим позже)
for col in ['region', 'city', 'store_age_cat', 'area_cat']:
    te = df.groupby(col)['rto'].transform('mean')
    df[f'te_{col}_mean'] = te
    te_med = df.groupby(col)['rto'].transform('median')
    df[f'te_{col}_median'] = te_med

# =========================
# 5) Проверка
# =========================
new_cols_block2 = [
    'rto_per_capita', 'rto_per_household', 'rto_per_kassa',
    'pyaterochka_share', 'competitors_per_capita',
    'traffic_total', 'traffic_per_kassa', 'foot_traffic_share',
    'infra_score', 'log1p_infra_score', 'infra_per_capita',
    'area_x_traffic', 'area_x_kass', 'kass_per_area',
    'region_rto_median', 'region_rto_mean', 'region_rto_std',
    'region_rto_dev', 'region_rto_rank',
    'city_rto_median', 'city_rto_mean', 'city_rto_std',
    'city_rto_dev', 'city_rto_rank',
    'region_month_median', 'region_month_dev',
    'te_region_mean', 'te_region_median',
    'te_city_mean', 'te_city_median',
    'te_store_age_cat_mean', 'te_store_age_cat_median',
    'te_area_cat_mean', 'te_area_cat_median',
]

print(f"Новых признаков в блоке 2: {len(new_cols_block2)}")
print(f"Общее число колонок: {df.shape[1]}")
print(f"\nПримеры значений (магазин 0, последняя дата):")
print(df[df['new_id'] == 0][new_cols_block2].tail(1).T)

Новых признаков в блоке 2: 34
Общее число колонок: 216

Примеры значений (магазин 0, последняя дата):
                                   25
rto_per_capita           1.442752e+02
rto_per_household        2.307349e+04
rto_per_kassa            7.920501e+06
pyaterochka_share        2.500000e-01
competitors_per_capita   4.967842e-06
traffic_total            2.110000e+02
traffic_per_kassa        1.918182e+01
foot_traffic_share       6.509434e-01
infra_score              1.000000e+00
log1p_infra_score        6.931472e-01
infra_per_capita         1.655947e-06
area_x_traffic           4.220000e+02
area_x_kass              2.000000e+01
kass_per_area            3.333333e+00
region_rto_median        6.874417e+07
region_rto_mean          7.696343e+07
region_rto_std           3.141544e+07
region_rto_dev           1.838134e+07
region_rto_rank          7.193676e-01
city_rto_median          6.669028e+07
city_rto_mean            7.550260e+07
city_rto_std             3.212410e+07
city_rto_dev            

In [4]:
from sklearn.model_selection import TimeSeriesSplit

# =========================
# 1) Подготовка таргета
# =========================

# Сначала создадим колонку с РТО за тот же месяц прошлого года
df['rto_same_month_last_year'] = df.groupby('new_id')['rto'].shift(12)

# Вариант 1: year-over-year ratio (оптимален для MAPE)
df['target_ratio'] = df['rto'] / (df['rto_same_month_last_year'] + 1e-6)

# Вариант 2: month-over-month ratio
df['target_mom_ratio'] = df['rto'] / (df['rto_lag_1'] + 1e-6)

# Вариант 3: log transformation (классический)
df['target_log'] = np.log1p(df['rto'])

# Также сохраним абсолютный таргет для MAPE
df['target_absolute'] = df['rto']

# =========================
# 2) Определяем период обучения
# =========================

# Чтобы использовать lag_12, нужна история с января 2023
# Обучаем с января 2024 по февраль 2025 (14 месяцев)
train_mask = (df['date'] >= '2024-01-01') & (df['date'] <= '2025-02-01')
train_df = df[train_mask].copy()

print(f"Обучающих строк: {len(train_df)}")
print(f"Период: {train_df['date'].min()} - {train_df['date'].max()}")
print(f"Уникальных магазинов: {train_df['new_id'].nunique()}")

# =========================
# 3) TimeSeriesSplit валидация
# =========================

# Разделяем по времени, чтобы имитировать реальное прогнозирование
# 4 фолда: валидация на каждом последнем месяце
dates = sorted(train_df['date'].unique())
print(f"\nВсе даты в train: {[str(d.date()) for d in dates]}")

# Создадим 4 фолда, где валидация = последний месяц
tscv_folds = []
for i in range(1, 5):
    val_date = dates[-i]
    train_dates = dates[:-i]

    train_idx = train_df[train_df['date'].isin(train_dates)].index
    val_idx = train_df[train_df['date'] == val_date].index

    tscv_folds.append((train_idx, val_idx))

    print(f"Фолд {4-i+1}: train {len(train_idx)} строк, val {len(val_idx)} строк на {val_date.date()}")

# =========================
# 4) Формирование тестового набора (март 2025)
# =========================

# Создаем DataFrame для теста
test_df = df[df['new_id'].isin(df['new_id'].unique())].drop_duplicates('new_id')[['new_id']].copy()
test_df['date'] = pd.Timestamp('2025-03-01')
test_df['year'] = 2025
test_df['month'] = 3

print(f"\nТестовых строк: {len(test_df)}")

# =========================
# 5) Подготовка фичей для теста
# =========================

# Для каждого магазина в тесте нужно взять актуальные лаги из последних данных
# Получаем последнюю строку каждого магазина из исходных данных
last_records = df.sort_values('date').groupby('new_id').last().reset_index()

# Копируем нужные признаки в test_df
feature_cols_to_copy = [
    # Статические признаки
    'store_age_cat', 'area_cat', 'city', 'region',
    'population', 'households', 'foot_traffic', 'car_traffic',
    'marketplaces', 'medical', 'schools', 'stops',
    'grocery_competitors', 'pyaterochka_nearby', 'n_kass', 'alcohol_flag',

    # Кодированные версии
    'store_age_ord', 'area_ord', 'city_freq', 'region_freq',

    # Логарифмы
    'log1p_population', 'log1p_households', 'log1p_foot_traffic',
    'log1p_car_traffic', 'log1p_marketplaces', 'log1p_medical',
    'log1p_schools', 'log1p_stops', 'log1p_grocery_competitors',
    'log1p_pyaterochka_nearby', 'log1p_n_kass',

    # Календарные (для марта 2025)
    'quarter', 'month_sin', 'month_cos', 'is_december', 'is_q4'
]

# Копируем статические признаки
for col in feature_cols_to_copy:
    if col in last_records.columns:
        test_df[col] = test_df['new_id'].map(last_records.set_index('new_id')[col])

# Обновляем календарные признаки для марта 2025
test_df['quarter'] = 1  # март - Q1
test_df['month_sin'] = np.sin(2 * np.pi * 3 / 12)
test_df['month_cos'] = np.cos(2 * np.pi * 3 / 12)
test_df['is_december'] = 0
test_df['is_q4'] = 0

# =========================
# 6) Функция для создания лагов в тесте
# =========================

def create_test_lags(test_df, full_df):
    """Создает лаги для тестового набора (март 2025)"""

    test_df = test_df.copy()

    # Для каждого магазина в тесте берем значения из последних доступных месяцев
    for col in ['rto', 'avg_promo', 'avg_items', 'avg_cancels', 'work_hours']:
        # Получаем историю магазина
        store_history = full_df[full_df['new_id'].isin(test_df['new_id'])]

        # lag_1 = февраль 2025
        lag1_vals = store_history[store_history['date'] == '2025-02-01'][['new_id', col]]
        lag1_vals = lag1_vals.rename(columns={col: f'{col}_lag_1'})
        test_df = test_df.merge(lag1_vals, on='new_id', how='left')

        # lag_2 = январь 2025
        lag2_vals = store_history[store_history['date'] == '2025-01-01'][['new_id', col]]
        lag2_vals = lag2_vals.rename(columns={col: f'{col}_lag_2'})
        test_df = test_df.merge(lag2_vals, on='new_id', how='left')

        # lag_3 = декабрь 2024
        lag3_vals = store_history[store_history['date'] == '2024-12-01'][['new_id', col]]
        lag3_vals = lag3_vals.rename(columns={col: f'{col}_lag_3'})
        test_df = test_df.merge(lag3_vals, on='new_id', how='left')

        # lag_12 = март 2024
        lag12_vals = store_history[store_history['date'] == '2024-03-01'][['new_id', col]]
        lag12_vals = lag12_vals.rename(columns={col: f'{col}_lag_12'})
        test_df = test_df.merge(lag12_vals, on='new_id', how='left')

        # Rolling mean 3 (ноябрь 2024 - январь 2025)
        # Считаем вручную
        for win_name, months in [('3', ['2024-11-01', '2024-12-01', '2025-01-01']),
                                 ('6', ['2024-08-01', '2024-09-01', '2024-10-01',
                                        '2024-11-01', '2024-12-01', '2025-01-01'])]:

            win_vals = store_history[store_history['date'].isin(months)].groupby('new_id')[col].mean().reset_index()
            win_vals = win_vals.rename(columns={col: f'{col}_roll_mean_{win_name}'})
            test_df = test_df.merge(win_vals, on='new_id', how='left')

    return test_df

# Создаем лаги для теста
test_df = create_test_lags(test_df, df)

print(f"\nТестовый набор после создания лагов:")
print(f"Размер: {test_df.shape}")
print(f"Колонки (первые 20): {list(test_df.columns[:20])}")

# =========================
# 7) Проверка данных
# =========================

print("\n=== Проверка обучающих данных ===")
print(f"Пропуски в target_ratio: {train_df['target_ratio'].isna().sum()}")
print(f"Пропуски в rto_lag_1: {train_df['rto_lag_1'].isna().sum()}")
print(f"Пропуски в rto_lag_12: {train_df['rto_lag_12'].isna().sum()}")

print("\n=== Статистика таргетов ===")
for target in ['target_ratio', 'target_mom_ratio', 'target_log']:
    print(f"{target}: mean={train_df[target].mean():.4f}, std={train_df[target].std():.4f}")

print("\n=== Пример одного магазина (new_id=0) ===")
store_example = train_df[train_df['new_id'] == 0].sort_values('date')
print(store_example[['date', 'rto', 'rto_same_month_last_year', 'target_ratio']].tail())

print("\n=== Тестовый пример (первый магазин) ===")
print(test_df.iloc[0][['new_id', 'date', 'rto_lag_1', 'rto_lag_12',
                       'avg_promo_lag_1', 'avg_items_lag_1']].to_dict())

Обучающих строк: 261198
Период: 2024-01-01 00:00:00 - 2025-02-01 00:00:00
Уникальных магазинов: 18657

Все даты в train: ['2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01', '2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01', '2025-01-01', '2025-02-01']
Фолд 4: train 242541 строк, val 18657 строк на 2025-02-01
Фолд 3: train 223884 строк, val 18657 строк на 2025-01-01
Фолд 2: train 205227 строк, val 18657 строк на 2024-12-01
Фолд 1: train 186570 строк, val 18657 строк на 2024-11-01

Тестовых строк: 18657

Тестовый набор после создания лагов:
Размер: (18657, 70)
Колонки (первые 20): ['new_id', 'date', 'year', 'month', 'store_age_cat', 'area_cat', 'city', 'region', 'population', 'households', 'foot_traffic', 'car_traffic', 'marketplaces', 'medical', 'schools', 'stops', 'grocery_competitors', 'pyaterochka_nearby', 'n_kass', 'alcohol_flag']

=== Проверка обучающих данных ===
Пропуски в target_ratio: 0
Пропуски в rto_lag_1: 0
Пр

In [5]:
eps = 1e-6

# =========================================================
# 1) Пересобираем test_df для марта 2025 ПРАВИЛЬНО
#    Берем февральский срез как базу статических признаков
#    и достраиваем лаги/роллинги из полной истории
# =========================================================

def build_march_test_from_history(df):
    feb_date = pd.Timestamp('2025-02-01')
    march_date = pd.Timestamp('2025-03-01')

    static_cols = [
        'new_id',
        'store_age_cat', 'area_cat', 'city', 'region',
        'population', 'households',
        'foot_traffic', 'car_traffic',
        'marketplaces', 'medical', 'schools', 'stops',
        'grocery_competitors', 'pyaterochka_nearby',
        'n_kass', 'alcohol_flag',
        'store_age_ord', 'area_ord',
        'city_freq', 'region_freq',
        'log1p_population', 'log1p_households',
        'log1p_foot_traffic', 'log1p_car_traffic',
        'log1p_marketplaces', 'log1p_medical',
        'log1p_schools', 'log1p_stops',
        'log1p_grocery_competitors', 'log1p_pyaterochka_nearby',
        'log1p_n_kass'
    ]

    test_df = (
        df[df['date'] == feb_date][static_cols]
        .drop_duplicates('new_id')
        .copy()
        .reset_index(drop=True)
    )

    # Календарь для марта 2025
    test_df['date'] = march_date
    test_df['year'] = 2025
    test_df['month'] = 3
    test_df['time_idx'] = (2025 - df['Год'].min()) * 12 + (3 - 1)
    test_df['quarter'] = 1
    test_df['month_sin'] = np.sin(2 * np.pi * 3 / 12)
    test_df['month_cos'] = np.cos(2 * np.pi * 3 / 12)
    test_df['is_december'] = 0
    test_df['is_q4'] = 0

    dyn_cols = ['rto', 'avg_promo', 'avg_items', 'avg_cancels', 'work_hours']

    lag_dates = {
        1: pd.Timestamp('2025-02-01'),
        2: pd.Timestamp('2025-01-01'),
        3: pd.Timestamp('2024-12-01'),
        6: pd.Timestamp('2024-09-01'),
        12: pd.Timestamp('2024-03-01'),
    }

    # Даты для rolling-окон для марта 2025:
    # roll_3  = Dec 2024, Jan 2025, Feb 2025
    # roll_6  = Sep 2024 ... Feb 2025
    # roll_12 = Mar 2024 ... Feb 2025
    rolling_dates = {
        3: pd.date_range(end=pd.Timestamp('2025-02-01'), periods=3, freq='MS'),
        6: pd.date_range(end=pd.Timestamp('2025-02-01'), periods=6, freq='MS'),
        12: pd.date_range(end=pd.Timestamp('2025-02-01'), periods=12, freq='MS'),
    }

    for col in dyn_cols:
        pivot = df.pivot(index='new_id', columns='date', values=col)
        pivot = pivot.reindex(index=test_df['new_id'])

        # Лаги
        for lag, lag_date in lag_dates.items():
            test_df[f'{col}_lag_{lag}'] = pivot[lag_date].values

        # Rolling statistics
        for win, dates in rolling_dates.items():
            mat = pivot.reindex(columns=dates)

            test_df[f'{col}_roll_mean_{win}'] = mat.mean(axis=1).values
            test_df[f'{col}_roll_std_{win}'] = mat.std(axis=1, ddof=0).values
            test_df[f'{col}_roll_median_{win}'] = mat.median(axis=1).values
            test_df[f'{col}_roll_min_{win}'] = mat.min(axis=1).values
            test_df[f'{col}_roll_max_{win}'] = mat.max(axis=1).values

    # =========================================================
    # 2) Derived-features для RTO
    # =========================================================
    test_df['rto_diff_1'] = test_df['rto_lag_1'] - test_df['rto_lag_2']
    test_df['rto_diff_3'] = test_df['rto_lag_1'] - test_df['rto_lag_3']
    test_df['rto_diff_12'] = test_df['rto_lag_1'] - test_df['rto_lag_12']

    test_df['rto_pct_change_1'] = test_df['rto_diff_1'] / (test_df['rto_lag_2'].abs() + eps)
    test_df['rto_pct_change_12'] = test_df['rto_diff_12'] / (test_df['rto_lag_12'].abs() + eps)

    test_df['rto_ratio_to_3m_mean'] = test_df['rto_lag_1'] / (test_df['rto_roll_mean_3'] + eps)
    test_df['rto_ratio_to_6m_mean'] = test_df['rto_lag_1'] / (test_df['rto_roll_mean_6'] + eps)
    test_df['rto_ratio_to_12m_mean'] = test_df['rto_lag_1'] / (test_df['rto_roll_mean_12'] + eps)

    test_df['rto_ratio_mom'] = test_df['rto_lag_1'] / (test_df['rto_lag_2'] + eps)
    test_df['rto_ratio_yoy'] = test_df['rto_lag_1'] / (test_df['rto_lag_12'] + eps)

    # =========================================================
    # 3) Derived-features для операционных метрик
    # =========================================================
    for col in ['avg_promo', 'avg_items', 'avg_cancels', 'work_hours']:
        test_df[f'{col}_mom_ratio'] = test_df[f'{col}_lag_1'] / (test_df[f'{col}_lag_2'] + eps)
        test_df[f'{col}_yoy_ratio'] = test_df[f'{col}_lag_1'] / (test_df[f'{col}_lag_12'] + eps)
        test_df[f'{col}_to_3m_mean'] = test_df[f'{col}_lag_1'] / (test_df[f'{col}_roll_mean_3'] + eps)

    # =========================================================
    # 4) Гео/социальные взаимодействия
    # =========================================================
    test_df['rto_per_capita'] = test_df['rto_lag_1'] / (test_df['population'] + 1)
    test_df['rto_per_household'] = test_df['rto_lag_1'] / (test_df['households'] + 1)
    test_df['rto_per_kassa'] = test_df['rto_lag_1'] / (test_df['n_kass'] + 1)

    test_df['pyaterochka_share'] = test_df['pyaterochka_nearby'] / (test_df['grocery_competitors'] + 1)
    test_df['competitors_per_capita'] = test_df['grocery_competitors'] / (test_df['population'] + 1)

    test_df['traffic_total'] = test_df['foot_traffic'] + test_df['car_traffic']
    test_df['traffic_per_kassa'] = test_df['traffic_total'] / (test_df['n_kass'] + 1)
    test_df['foot_traffic_share'] = test_df['foot_traffic'] / (test_df['traffic_total'] + 1)

    test_df['infra_score'] = (
        test_df['marketplaces'] + test_df['medical'] + test_df['schools'] + test_df['stops']
    )
    test_df['log1p_infra_score'] = np.log1p(test_df['infra_score'])
    test_df['infra_per_capita'] = test_df['infra_score'] / (test_df['population'] + 1)

    test_df['area_x_traffic'] = test_df['area_ord'] * test_df['traffic_total']
    test_df['area_x_kass'] = test_df['area_ord'] * test_df['n_kass']
    test_df['kass_per_area'] = test_df['n_kass'] / (test_df['area_ord'] + 1)

    # =========================================================
    # 5) Групповые агрегаты для марта 2025
    #    Они должны строиться на rto_lag_1 (= февраль 2025)
    # =========================================================
    for agg_col in ['region', 'city']:
        grp = test_df.groupby(agg_col)['rto_lag_1']

        test_df[f'{agg_col}_rto_median'] = grp.transform('median')
        test_df[f'{agg_col}_rto_mean'] = grp.transform('mean')
        test_df[f'{agg_col}_rto_std'] = grp.transform('std')
        test_df[f'{agg_col}_rto_dev'] = test_df['rto_lag_1'] - test_df[f'{agg_col}_rto_median']
        test_df[f'{agg_col}_rto_rank'] = grp.rank(pct=True)

    test_df['region_month_median'] = test_df.groupby(['region', 'month'])['rto_lag_1'].transform('median')
    test_df['region_month_dev'] = test_df['rto_lag_1'] - test_df['region_month_median']

    # Инфинити -> NaN
    test_df = test_df.replace([np.inf, -np.inf], np.nan)

    return test_df


# Пересобираем test
test_df = build_march_test_from_history(df)

print("test_df shape:", test_df.shape)
print("Пример колонок:", test_df.columns[:30].tolist())
print(test_df[['new_id', 'date', 'rto_lag_1', 'rto_lag_2', 'rto_lag_3', 'rto_lag_6', 'rto_lag_12']].head())

test_df shape: (18657, 189)
Пример колонок: ['new_id', 'store_age_cat', 'area_cat', 'city', 'region', 'population', 'households', 'foot_traffic', 'car_traffic', 'marketplaces', 'medical', 'schools', 'stops', 'grocery_competitors', 'pyaterochka_nearby', 'n_kass', 'alcohol_flag', 'store_age_ord', 'area_ord', 'city_freq', 'region_freq', 'log1p_population', 'log1p_households', 'log1p_foot_traffic', 'log1p_car_traffic', 'log1p_marketplaces', 'log1p_medical', 'log1p_schools', 'log1p_stops', 'log1p_grocery_competitors']
   new_id       date    rto_lag_1    rto_lag_2    rto_lag_3    rto_lag_6  \
0       0 2025-03-01  82659801.63  87125506.92  92481231.23  78413715.79   
1       1 2025-03-01  39802499.35  38727033.56  45602389.75  38344713.49   
2       2 2025-03-01  81370545.90  79225807.62  92130609.50  69826431.16   
3       3 2025-03-01  59854159.11  61778593.47  77159939.66  60093085.82   
4       4 2025-03-01  73849939.53  76891676.38  87932821.03  70664052.54   

    rto_lag_12  
0  7899

In [6]:
# =========================================================
# 6) Готовим train_df заново в правильном порядке индексов
# =========================================================

train_df = train_df.sort_values(['date', 'new_id']).reset_index(drop=True)

# На всякий случай убираем inf
train_df = train_df.replace([np.inf, -np.inf], np.nan)

print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)

train_df shape: (261198, 221)
test_df shape: (18657, 189)


In [7]:
# =========================================================
# 7) Исключаем leaky и недоступные на inference признаки
# =========================================================

# te_* пока НЕ используем — это leakage
te_cols = [c for c in train_df.columns if c.startswith('te_')]

drop_cols = {
    'rto',
    'target_ratio',
    'target_mom_ratio',
    'target_log',
    'target_absolute',
    'rto_same_month_last_year',
    'date',
    'Год',
    'Месяц',

    # текущие динамические признаки марта нам неизвестны
    'avg_promo',
    'avg_items',
    'avg_cancels',
    'work_hours',
}

drop_cols = drop_cols.union(set(te_cols))

# Оставляем только общие признаки между train и test
common_cols = sorted((set(train_df.columns) & set(test_df.columns)) - drop_cols)

# Отдельно:
# numeric features -> для LightGBM / XGBoost / Ridge / ExtraTrees
# cat features     -> для CatBoost
cat_features = ['city', 'region', 'store_age_cat', 'area_cat']
cat_features = [c for c in cat_features if c in common_cols]

numeric_features = [
    c for c in common_cols
    if c not in cat_features and pd.api.types.is_numeric_dtype(train_df[c])
]

catboost_features = numeric_features + cat_features

print("Количество numeric_features:", len(numeric_features))
print("Количество catboost_features:", len(catboost_features))
print("Категориальные для CatBoost:", cat_features[:])

Количество numeric_features: 184
Количество catboost_features: 188
Категориальные для CatBoost: ['city', 'region', 'store_age_cat', 'area_cat']


In [8]:
# =========================================================
# 8) Финальные матрицы
# =========================================================

X_train_num = train_df[numeric_features].copy()
X_test_num = test_df[numeric_features].copy()

X_train_cat = train_df[catboost_features].copy()
X_test_cat = test_df[catboost_features].copy()

y_ratio = train_df['target_ratio'].copy()
y_abs = train_df['target_absolute'].copy()
y_log = train_df['target_log'].copy()

print("X_train_num:", X_train_num.shape)
print("X_test_num :", X_test_num.shape)
print("X_train_cat:", X_train_cat.shape)
print("X_test_cat :", X_test_cat.shape)

X_train_num: (261198, 184)
X_test_num : (18657, 184)
X_train_cat: (261198, 188)
X_test_cat : (18657, 188)


In [9]:
# =========================================================
# 9) Создаем честные time-series folds
# =========================================================

val_dates = [
    pd.Timestamp('2024-11-01'),
    pd.Timestamp('2024-12-01'),
    pd.Timestamp('2025-01-01'),
    pd.Timestamp('2025-02-01'),
]

folds = []
for i, val_date in enumerate(val_dates, 1):
    tr_idx = np.where(train_df['date'] < val_date)[0]
    va_idx = np.where(train_df['date'] == val_date)[0]
    folds.append((tr_idx, va_idx))
    print(f'Fold {i}: train={len(tr_idx)}, valid={len(va_idx)}, val_date={val_date.date()}')

Fold 1: train=186570, valid=18657, val_date=2024-11-01
Fold 2: train=205227, valid=18657, val_date=2024-12-01
Fold 3: train=223884, valid=18657, val_date=2025-01-01
Fold 4: train=242541, valid=18657, val_date=2025-02-01


In [10]:
# =========================================================
# 10) Санити-чеки
# =========================================================

print("\n=== Топ пропусков в train numeric ===")
print(
    X_train_num.isna().mean().sort_values(ascending=False).head(20)
)

print("\n=== Топ пропусков в test numeric ===")
print(
    X_test_num.isna().mean().sort_values(ascending=False).head(20)
)

print("\n=== Проверка таргета ===")
print("target_ratio stats:")
print(y_ratio.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

print("\n=== Проверка тестовых лагов ===")
print(
    test_df[['rto_lag_1', 'rto_lag_2', 'rto_lag_3', 'rto_lag_6', 'rto_lag_12']].describe()
)


=== Топ пропусков в train numeric ===
city_rto_std                  0.095621
alcohol_flag                  0.000000
area_x_kass                   0.000000
area_ord                      0.000000
avg_cancels_lag_1             0.000000
avg_cancels_lag_12            0.000000
avg_cancels_lag_2             0.000000
area_x_traffic                0.000000
avg_cancels_lag_6             0.000000
avg_cancels_mom_ratio         0.000000
avg_cancels_roll_max_12       0.000000
avg_cancels_roll_max_3        0.000000
avg_cancels_roll_max_6        0.000000
avg_cancels_roll_mean_12      0.000000
avg_cancels_roll_mean_3       0.000000
avg_cancels_roll_mean_6       0.000000
avg_cancels_roll_median_12    0.000000
avg_cancels_roll_median_3     0.000000
avg_cancels_roll_median_6     0.000000
avg_cancels_roll_min_12       0.000000
dtype: float64

=== Топ пропусков в test numeric ===
city_rto_std                  0.095621
alcohol_flag                  0.000000
area_x_kass                   0.000000
area_ord   

In [11]:
import lightgbm as lgb

# ==========================================
# 1) Метрики и служебные функции
# ==========================================

EPS = 1e-6

def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs(y_pred - y_true) / np.clip(np.abs(y_true), EPS, None)) * 100

def competition_points(mape_value):
    return 100 * ((100 - min(mape_value, 100)) / 100) ** 2

# Добавим новые таргеты
train_df['target_log_ratio'] = np.log(train_df['target_ratio'].clip(lower=EPS))
train_df['target_log_abs2'] = np.log(train_df['target_absolute'].clip(lower=EPS))

print("target_log_ratio stats:")
print(train_df['target_log_ratio'].describe())

print("\ntarget_log_abs2 stats:")
print(train_df['target_log_abs2'].describe())

target_log_ratio stats:
count    261198.000000
mean          0.132881
std           0.146078
min          -2.071036
25%           0.060956
50%           0.133256
75%           0.204394
max           4.153849
Name: target_log_ratio, dtype: float64

target_log_abs2 stats:
count    261198.000000
mean         18.243111
std           0.464483
min          16.042457
25%          17.911349
50%          18.198015
75%          18.534716
max          20.224029
Name: target_log_abs2, dtype: float64


In [12]:
# ==========================================
# Обновляем run_lgb_oof с поддержкой mom_ratio
# ==========================================

def run_lgb_oof(
    X_train, X_test, train_df, test_df, folds,
    target_col, mode='log_ratio', params=None, model_name='lgb'
):
    if params is None:
        raise ValueError("Передай params")

    y = train_df[target_col].values

    oof_abs = np.full(len(X_train), np.nan)
    test_abs_folds = np.zeros((len(X_test), len(folds)))
    fold_scores = []

    for fold_num, (tr_idx, va_idx) in enumerate(folds, 1):
        X_tr = X_train.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]
        y_tr = y[tr_idx]
        y_va = y[va_idx]

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric='l1',
            callbacks=[
                lgb.early_stopping(stopping_rounds=300, verbose=False),
                lgb.log_evaluation(period=0)
            ]
        )

        pred_va = model.predict(X_va, num_iteration=model.best_iteration_)
        pred_test = model.predict(X_test, num_iteration=model.best_iteration_)

        if mode == 'log_ratio':
            pred_va_abs = np.exp(pred_va) * train_df.iloc[va_idx]['rto_lag_12'].values
            pred_test_abs = np.exp(pred_test) * test_df['rto_lag_12'].values
        elif mode == 'log_abs':
            pred_va_abs = np.exp(pred_va)
            pred_test_abs = np.exp(pred_test)
        elif mode == 'mom_ratio':
            pred_va_abs = pred_va * train_df.iloc[va_idx]['rto_lag_1'].values
            pred_test_abs = pred_test * test_df['rto_lag_1'].values

        pred_va_abs = np.clip(pred_va_abs, 1e5, None)
        pred_test_abs = np.clip(pred_test_abs, 1e5, None)

        true_va_abs = train_df.iloc[va_idx]['target_absolute'].values
        fold_mape = mape(true_va_abs, pred_va_abs)

        print(f"{model_name} | Fold {fold_num} | best_iter={model.best_iteration_} | MAPE={fold_mape:.4f}")

        fold_scores.append(fold_mape)
        oof_abs[va_idx] = pred_va_abs
        test_abs_folds[:, fold_num - 1] = pred_test_abs

    covered_mask = ~np.isnan(oof_abs)
    oof_mape = mape(
        train_df.loc[covered_mask, 'target_absolute'].values,
        oof_abs[covered_mask]
    )
    oof_points = competition_points(oof_mape)

    print(f"\n{model_name} FINAL OOF MAPE: {oof_mape:.5f}")
    print(f"{model_name} Mean fold MAPE: {np.mean(fold_scores):.5f}")

    return {
        'oof_abs': oof_abs,
        'covered_mask': covered_mask,
        'test_abs': test_abs_folds.mean(axis=1),
        'fold_scores': fold_scores,
        'oof_mape': oof_mape,
        'oof_points': oof_points,
    }

In [13]:
!pip install optuna

import optuna
from optuna.samplers import TPESampler

# Подавляем спам Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


def tune_lgb_optuna(
    X_train,
    train_df,
    folds,
    target_col,
    mode='log_ratio',
    n_trials=100,
    timeout=None,
    seed=42,
    study_name='lgb_tuning',
    verbose=True
):
    """
    Подбор гиперпараметров LightGBM через Optuna.

    Параметры:
    ----------
    X_train : pd.DataFrame
        Признаки
    train_df : pd.DataFrame
        Содержит target_absolute, rto_lag_12, rto_lag_1
    folds : list of (train_idx, val_idx)
        Time series folds
    target_col : str
        'target_log_ratio', 'target_log_abs2', 'target_mom_ratio'
    mode : str
        'log_ratio', 'log_abs', 'mom_ratio'
    n_trials : int
        Количество попыток (рекомендую 100-300)
    timeout : int or None
        Максимальное время в секундах
    seed : int
    study_name : str

    Возвращает:
    -----------
    best_params : dict
    study : optuna.Study
    """

    y = train_df[target_col].values

    def objective(trial):
        params = {
            'objective': 'regression',
            'metric': 'l1',
            'n_estimators': 10000,  # фиксируем большое, контролируем early_stopping
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, 255),
            'max_depth': trial.suggest_int('max_depth', 4, 12),
            'min_child_samples': trial.suggest_int('min_child_samples', 20, 300),
            'min_child_weight': trial.suggest_float('min_child_weight', 1e-3, 10.0, log=True),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'subsample_freq': trial.suggest_int('subsample_freq', 1, 7),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
            'random_state': seed,
            'n_jobs': -1,
            'verbose': -1,
        }

        fold_mapes = []

        for fold_num, (tr_idx, va_idx) in enumerate(folds, 1):
            X_tr = X_train.iloc[tr_idx]
            X_va = X_train.iloc[va_idx]
            y_tr = y[tr_idx]
            y_va = y[va_idx]

            model = lgb.LGBMRegressor(**params)

            model.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                eval_metric='l1',
                callbacks=[
                    lgb.early_stopping(stopping_rounds=200, verbose=False),
                    lgb.log_evaluation(period=0)
                ]
            )

            pred_va = model.predict(X_va, num_iteration=model.best_iteration_)

            # Обратное преобразование под нужный mode
            if mode == 'log_ratio':
                pred_va_abs = np.exp(pred_va) * train_df.iloc[va_idx]['rto_lag_12'].values
            elif mode == 'log_abs':
                pred_va_abs = np.exp(pred_va)
            elif mode == 'mom_ratio':
                pred_va_abs = pred_va * train_df.iloc[va_idx]['rto_lag_1'].values
            else:
                raise ValueError(f"Неизвестный mode: {mode}")

            pred_va_abs = np.clip(pred_va_abs, 1e5, None)

            true_va_abs = train_df.iloc[va_idx]['target_absolute'].values

            fold_mape = np.mean(
                np.abs(pred_va_abs - true_va_abs) / np.clip(np.abs(true_va_abs), 1e-6, None)
            ) * 100

            fold_mapes.append(fold_mape)

            # Пруннинг: если результат уже плохой после первого фолда — прерываем
            trial.report(np.mean(fold_mapes), step=fold_num)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        return np.mean(fold_mapes)

    sampler = TPESampler(seed=seed)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2)

    study = optuna.create_study(
        direction='minimize',
        sampler=sampler,
        pruner=pruner,
        study_name=study_name
    )

    if verbose:
        print(f"\n{'='*60}")
        print(f"OPTUNA TUNING: {study_name}")
        print(f"Mode: {mode}, Target: {target_col}")
        print(f"Trials: {n_trials}, Folds: {len(folds)}")
        print(f"{'='*60}\n")

    # Callback для логирования прогресса
    def logging_callback(study, trial):
        if verbose and trial.number % 10 == 0:
            print(f"  Trial {trial.number}: current best MAPE = {study.best_value:.5f}")

    study.optimize(
        objective,
        n_trials=n_trials,
        timeout=timeout,
        callbacks=[logging_callback],
        show_progress_bar=False
    )

    best_params = study.best_params.copy()
    best_params.update({
        'objective': 'regression',
        'metric': 'l1',
        'n_estimators': 10000,
        'random_state': seed,
        'n_jobs': -1,
        'verbose': -1,
    })

    if verbose:
        print(f"\n{'='*60}")
        print(f"BEST RESULT for {study_name}")
        print(f"{'='*60}")
        print(f"Best MAPE: {study.best_value:.5f}")
        print(f"Best params:")
        for k, v in study.best_params.items():
            print(f"  {k}: {v}")

    return best_params, study

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.7 MB/s eta 0:00:00


In [14]:
# ==========================================
# 1) Тюнинг LGB_log_ratio
# ==========================================

best_params_logratio, study_logratio = tune_lgb_optuna(
    X_train=X_train_num,
    train_df=train_df,
    folds=folds,
    target_col='target_log_ratio',
    mode='log_ratio',
    n_trials=30,
    seed=42,
    study_name='LGB_log_ratio'
)


OPTUNA TUNING: LGB_log_ratio
Mode: log_ratio, Target: target_log_ratio
Trials: 30, Folds: 4

  Trial 0: current best MAPE = 5.32312
  Trial 10: current best MAPE = 5.32312
  Trial 20: current best MAPE = 5.28258

BEST RESULT for LGB_log_ratio
Best MAPE: 5.16370
Best params:
  learning_rate: 0.005160968768096091
  num_leaves: 54
  max_depth: 8
  min_child_samples: 49
  min_child_weight: 0.009535400814595932
  subsample: 0.9233556350190333
  subsample_freq: 4
  colsample_bytree: 0.7926568065394248
  reg_alpha: 0.31423869800216053
  reg_lambda: 4.7942976434286
  min_split_gain: 0.0015407987938578227


In [15]:
# ==========================================
# 2) Тюнинг LGB_log_abs
# ==========================================

best_params_logabs, study_logabs = tune_lgb_optuna(
    X_train=X_train_num,
    train_df=train_df,
    folds=folds,
    target_col='target_log_abs2',
    mode='log_abs',
    n_trials=30,
    seed=42,
    study_name='LGB_log_abs'
)


OPTUNA TUNING: LGB_log_abs
Mode: log_abs, Target: target_log_abs2
Trials: 30, Folds: 4

  Trial 0: current best MAPE = 6.88170
  Trial 10: current best MAPE = 6.50941
  Trial 20: current best MAPE = 6.48538

BEST RESULT for LGB_log_abs
Best MAPE: 6.48538
Best params:
  learning_rate: 0.015610427288202826
  num_leaves: 18
  max_depth: 7
  min_child_samples: 87
  min_child_weight: 0.7433592390017701
  subsample: 0.9074147324713179
  subsample_freq: 2
  colsample_bytree: 0.9855979142409212
  reg_alpha: 1.393951639987
  reg_lambda: 0.09276940295031436
  min_split_gain: 0.19786216075101665


In [16]:
# ==========================================
# 3) Тюнинг LGB_mom_ratio
# ==========================================

best_params_momratio, study_momratio = tune_lgb_optuna(
    X_train=X_train_num,
    train_df=train_df,
    folds=folds,
    target_col='target_mom_ratio',
    mode='mom_ratio',
    n_trials=30,
    seed=42,
    study_name='LGB_mom_ratio'
)


OPTUNA TUNING: LGB_mom_ratio
Mode: mom_ratio, Target: target_mom_ratio
Trials: 30, Folds: 4

  Trial 0: current best MAPE = 6.67948
  Trial 10: current best MAPE = 6.52285
  Trial 20: current best MAPE = 6.52285

BEST RESULT for LGB_mom_ratio
Best MAPE: 6.52285
Best params:
  learning_rate: 0.022989844601573283
  num_leaves: 16
  max_depth: 7
  min_child_samples: 25
  min_child_weight: 0.021206551805691195
  subsample: 0.8951735669692686
  subsample_freq: 3
  colsample_bytree: 0.982458150673019
  reg_alpha: 0.0018066338345909892
  reg_lambda: 0.03299263281612611
  min_split_gain: 0.2807609542397041


In [17]:
# ==========================================
# 4) Переобучение моделей с лучшими параметрами
# ==========================================

lgb_logratio_tuned = run_lgb_oof(
    X_train=X_train_num,
    X_test=X_test_num,
    train_df=train_df,
    test_df=test_df,
    folds=folds,
    target_col='target_log_ratio',
    mode='log_ratio',
    params=best_params_logratio,
    model_name='LGB_log_ratio_TUNED'
)

lgb_logabs_tuned = run_lgb_oof(
    X_train=X_train_num,
    X_test=X_test_num,
    train_df=train_df,
    test_df=test_df,
    folds=folds,
    target_col='target_log_abs2',
    mode='log_abs',
    params=best_params_logabs,
    model_name='LGB_log_abs_TUNED'
)

lgb_momratio_tuned = run_lgb_oof(
    X_train=X_train_num,
    X_test=X_test_num,
    train_df=train_df,
    test_df=test_df,
    folds=folds,
    target_col='target_mom_ratio',
    mode='mom_ratio',
    params=best_params_momratio,
    model_name='LGB_mom_ratio_TUNED'
)

LGB_log_ratio_TUNED | Fold 1 | best_iter=2868 | MAPE=3.6792
LGB_log_ratio_TUNED | Fold 2 | best_iter=93 | MAPE=7.1183
LGB_log_ratio_TUNED | Fold 3 | best_iter=9983 | MAPE=4.2682
LGB_log_ratio_TUNED | Fold 4 | best_iter=1044 | MAPE=5.5116

LGB_log_ratio_TUNED FINAL OOF MAPE: 5.14434
LGB_log_ratio_TUNED Mean fold MAPE: 5.14434
LGB_log_abs_TUNED | Fold 1 | best_iter=687 | MAPE=3.8913
LGB_log_abs_TUNED | Fold 2 | best_iter=243 | MAPE=11.4653
LGB_log_abs_TUNED | Fold 3 | best_iter=741 | MAPE=4.3030
LGB_log_abs_TUNED | Fold 4 | best_iter=734 | MAPE=6.2818

LGB_log_abs_TUNED FINAL OOF MAPE: 6.48538
LGB_log_abs_TUNED Mean fold MAPE: 6.48538
LGB_mom_ratio_TUNED | Fold 1 | best_iter=78 | MAPE=3.8781
LGB_mom_ratio_TUNED | Fold 2 | best_iter=59 | MAPE=12.3102
LGB_mom_ratio_TUNED | Fold 3 | best_iter=183 | MAPE=4.2478
LGB_mom_ratio_TUNED | Fold 4 | best_iter=6 | MAPE=5.6554

LGB_mom_ratio_TUNED FINAL OOF MAPE: 6.52285
LGB_mom_ratio_TUNED Mean fold MAPE: 6.52285


In [23]:
comparison = pd.DataFrame({
    'model': [
        'LGB_log_ratio (TUNED)',
        'LGB_log_abs (TUNED)',
        'LGB_mom_ratio (TUNED)'
    ],
    'oof_mape': [
        lgb_logratio_tuned['oof_mape'],
        lgb_logabs_tuned['oof_mape'],
        lgb_momratio_tuned['oof_mape'],
    ]
})

print(comparison)

                   model  oof_mape
0  LGB_log_ratio (TUNED)  5.144341
1    LGB_log_abs (TUNED)  6.485385
2  LGB_mom_ratio (TUNED)  6.522853


In [19]:
# ==========================================
# 6) Сохранение лучших параметров (на всякий случай)
# ==========================================

import json

all_best_params = {
    'log_ratio': best_params_logratio,
    'log_abs': best_params_logabs,
    'mom_ratio': best_params_momratio
}

# Сериализуем (только то, что можно)
all_best_params_serializable = {}
for name, params in all_best_params.items():
    all_best_params_serializable[name] = {
        k: v for k, v in params.items() if isinstance(v, (int, float, str, bool, type(None)))
    }

with open('best_lgb_params.json', 'w') as f:
    json.dump(all_best_params_serializable, f, indent=2)

print("Параметры сохранены в best_lgb_params.json")

Параметры сохранены в best_lgb_params.json


In [24]:
from scipy.optimize import minimize

# ==========================================
# 1) Подготовка OOF и test матриц
# ==========================================

results_list = [
    ('LGB_log_ratio_TUNED', lgb_logratio_tuned),
    ('LGB_log_abs_TUNED', lgb_logabs_tuned),
    ('LGB_mom_ratio_TUNED', lgb_momratio_tuned),
]

model_names = [r[0] for r in results_list]
results = [r[1] for r in results_list]

# Общий mask
all_masks = [r['covered_mask'] for r in results]
common_mask = np.logical_and.reduce(all_masks)

# OOF матрица
oof_preds = np.column_stack([r['oof_abs'][common_mask] for r in results])

# Test матрица
test_preds = np.column_stack([r['test_abs'] for r in results])

y_true_oof = train_df.loc[common_mask, 'target_absolute'].values

print(f"OOF матрица: {oof_preds.shape}")
print(f"Test матрица: {test_preds.shape}")
print(f"Покрыто OOF: {common_mask.sum()} строк")

OOF матрица: (74628, 3)
Test матрица: (18657, 3)
Покрыто OOF: 74628 строк


In [25]:
# ==========================================
# 2) Корреляция OOF-предсказаний
# ==========================================

oof_df = pd.DataFrame(oof_preds, columns=model_names)
print("\nКорреляции OOF:")
print(oof_df.corr().round(4))

print("\nMAPE каждой модели:")
for name, r in results_list:
    print(f"  {name}: {r['oof_mape']:.5f}")


Корреляции OOF:
                     LGB_log_ratio_TUNED  LGB_log_abs_TUNED  \
LGB_log_ratio_TUNED               1.0000             0.9911   
LGB_log_abs_TUNED                 0.9911             1.0000   
LGB_mom_ratio_TUNED               0.9905             0.9965   

                     LGB_mom_ratio_TUNED  
LGB_log_ratio_TUNED               0.9905  
LGB_log_abs_TUNED                 0.9965  
LGB_mom_ratio_TUNED               1.0000  

MAPE каждой модели:
  LGB_log_ratio_TUNED: 5.14434
  LGB_log_abs_TUNED: 6.48538
  LGB_mom_ratio_TUNED: 6.52285


In [28]:
# ==========================================
# 3) Метрики
# ==========================================

EPS = 1e-6

def mape(y_true, y_pred):
    return np.mean(np.abs(y_pred - y_true) / np.clip(np.abs(y_true), EPS, None)) * 100

def competition_points(mape_value):
    return 100 * ((100 - min(mape_value, 100)) / 100) ** 2

In [29]:
# ==========================================
# 4) Оптимизация весов через scipy
# ==========================================

def blend_mape_n(weights, preds, y_true):
    weights = np.abs(weights)
    weights = weights / weights.sum()
    blend = preds @ weights
    return mape(y_true, blend)

n_models = oof_preds.shape[1]

# Множественные старты
best_result = None
best_score = float('inf')

starting_points = [
    np.ones(n_models) / n_models,             # равные веса
    np.array([1.0, 0.0, 0.0]),                # только лучшая
    np.array([0.8, 0.1, 0.1]),                # доминанта
    np.array([0.7, 0.15, 0.15]),
    np.array([0.6, 0.2, 0.2]),
    np.array([0.5, 0.25, 0.25]),
    np.array([0.9, 0.05, 0.05]),
]

for x0 in starting_points:
    result = minimize(
        blend_mape_n,
        x0=x0,
        args=(oof_preds, y_true_oof),
        method='Nelder-Mead',
        options={'maxiter': 3000, 'xatol': 1e-7, 'fatol': 1e-7}
    )
    if result.fun < best_score:
        best_score = result.fun
        best_result = result

optimal_weights = np.abs(best_result.x) / np.abs(best_result.x).sum()

print("\n=== Оптимальные веса ансамбля ===")
for name, w in zip(model_names, optimal_weights):
    print(f"  {name:25s}: {w:.4f}")

print(f"\nBlend OOF MAPE: {best_score:.5f}")
print(f"Blend OOF Points: {competition_points(best_score):.5f}")
print(f"Лучшая single: {min(r['oof_mape'] for _, r in results_list):.5f}")
print(f"Прирост: {min(r['oof_mape'] for _, r in results_list) - best_score:.5f}")


=== Оптимальные веса ансамбля ===
  LGB_log_ratio_TUNED      : 0.8656
  LGB_log_abs_TUNED        : 0.0000
  LGB_mom_ratio_TUNED      : 0.1344

Blend OOF MAPE: 5.10723
Blend OOF Points: 90.04637
Лучшая single: 5.14434
Прирост: 0.03711


In [30]:
# ==========================================
# 5) Альтернатива: равные веса (часто стабильнее на LB)
# ==========================================

equal_weights = np.ones(n_models) / n_models
equal_blend_oof = oof_preds @ equal_weights
equal_mape = mape(y_true_oof, equal_blend_oof)

print(f"\n=== Равные веса (для сравнения) ===")
print(f"Equal blend MAPE: {equal_mape:.5f}")
print(f"Equal blend Points: {competition_points(equal_mape):.5f}")


=== Равные веса (для сравнения) ===
Equal blend MAPE: 5.67908
Equal blend Points: 88.96436


In [31]:
# ==========================================
# 6) Альтернатива: ограниченные веса (минимум 0.05 на каждую)
#    Часто лучше generalizes на LB
# ==========================================

from scipy.optimize import minimize

def blend_mape_constrained(weights, preds, y_true):
    weights = weights / weights.sum()
    blend = preds @ weights
    return mape(y_true, blend)

constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}]
bounds = [(0.05, 0.95)] * n_models  # каждая модель имеет минимум 5%

result_constrained = minimize(
    blend_mape_constrained,
    x0=np.ones(n_models) / n_models,
    args=(oof_preds, y_true_oof),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 1000, 'ftol': 1e-8}
)

constrained_weights = result_constrained.x / result_constrained.x.sum()
constrained_mape = result_constrained.fun

print(f"\n=== Ограниченные веса (min 5% на модель) ===")
for name, w in zip(model_names, constrained_weights):
    print(f"  {name:25s}: {w:.4f}")
print(f"Constrained blend MAPE: {constrained_mape:.5f}")
print(f"Constrained blend Points: {competition_points(constrained_mape):.5f}")


=== Ограниченные веса (min 5% на модель) ===
  LGB_log_ratio_TUNED      : 0.8563
  LGB_log_abs_TUNED        : 0.0500
  LGB_mom_ratio_TUNED      : 0.0937
Constrained blend MAPE: 5.11670
Constrained blend Points: 90.02841


In [32]:
# ==========================================
# 7) Итоговое сравнение всех вариантов
# ==========================================

summary = pd.DataFrame({
    'strategy': [
        'Single LGB_log_ratio (TUNED)',
        'Equal weights (1/3 каждой)',
        'Optimal weights (scipy)',
        'Constrained weights (min 5%)',
    ],
    'oof_mape': [
        lgb_logratio_tuned['oof_mape'],
        equal_mape,
        best_score,
        constrained_mape,
    ],
    'oof_points': [
        competition_points(lgb_logratio_tuned['oof_mape']),
        competition_points(equal_mape),
        competition_points(best_score),
        competition_points(constrained_mape),
    ]
}).sort_values('oof_mape')

print("\n=== ИТОГОВОЕ СРАВНЕНИЕ ===")
print(summary.to_string(index=False))


=== ИТОГОВОЕ СРАВНЕНИЕ ===
                    strategy  oof_mape  oof_points
     Optimal weights (scipy)  5.107235   90.046369
Constrained weights (min 5%)  5.116697   90.028412
Single LGB_log_ratio (TUNED)  5.144341   89.975961
  Equal weights (1/3 каждой)  5.679077   88.964364


In [33]:
# ==========================================
# 8) Создаём 3 разных submission для проверки на LB
# ==========================================

# Вариант 1: только лучшая single модель
test_single = lgb_logratio_tuned['test_abs']

# Вариант 2: optimal weights
test_optimal = test_preds @ optimal_weights

# Вариант 3: constrained weights
test_constrained = test_preds @ constrained_weights

# Вариант 4: equal weights
test_equal = test_preds @ equal_weights

submissions = {
    'submission_single_lgb.csv': test_single,
    'submission_optimal_weights.csv': test_optimal,
    'submission_constrained_weights.csv': test_constrained,
    'submission_equal_weights.csv': test_equal,
}

for filename, preds in submissions.items():
    sub = pd.DataFrame({
        'new_id': test_df['new_id'],
        'РТО': preds
    })
    sub.to_csv(filename, index=False)
    print(f"\n{filename}:")
    print(f"  mean={sub['РТО'].mean():.2e}, median={sub['РТО'].median():.2e}")
    print(f"  min={sub['РТО'].min():.2e}, max={sub['РТО'].max():.2e}")


submission_single_lgb.csv:
  mean=1.01e+08, median=8.61e+07
  min=1.52e+07, max=5.29e+08

submission_optimal_weights.csv:
  mean=1.01e+08, median=8.58e+07
  min=1.51e+07, max=5.28e+08

submission_constrained_weights.csv:
  mean=1.01e+08, median=8.59e+07
  min=1.56e+07, max=5.20e+08

submission_equal_weights.csv:
  mean=1.00e+08, median=8.53e+07
  min=1.78e+07, max=4.75e+08


In [34]:
# ==========================================
# 9) Sanity check: распределение предсказаний vs март 2024
# ==========================================

march_2024 = df[df['date'] == '2024-03-01']['rto']

print("\n=== Sanity check: распределение РТО ===")
print(f"Март 2024 (исторический): mean={march_2024.mean():.2e}, median={march_2024.median():.2e}")
print(f"Март 2025 (predict best):  mean={test_optimal.mean():.2e}, median={np.median(test_optimal):.2e}")

expected_growth = test_optimal.mean() / march_2024.mean() - 1
print(f"\nОжидаемый годовой рост: {expected_growth*100:.2f}%")
print("(норма: 5-15%, тревога если <0% или >25%)")


=== Sanity check: распределение РТО ===
Март 2024 (исторический): mean=9.38e+07, median=7.94e+07
Март 2025 (predict best):  mean=1.01e+08, median=8.58e+07

Ожидаемый годовой рост: 7.64%
(норма: 5-15%, тревога если <0% или >25%)


In [20]:
# submission_lgb = pd.DataFrame({
#     'new_id': test_df['new_id'],
#     'rto': best_lgb_result['test_abs']
# })

# print(submission_lgb.head())
# print(submission_lgb['rto'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))